In [2]:
"""
BigAlpha 2026 因子挖掘 — 财务因子 ROE_TTM（传统量化赛道）

官方要求：import、辅助函数、全部逻辑均须在 main() 内。
参考：https://bigquant.com/wiki/doc/m8APtGyV9f
"""


def main(datasource, start_date, end_date):
    # 【经济逻辑】ROE_TTM = 归母净利润(TTM) / 归母净资产(LF)，衡量股东资本回报效率
    # 【统计方法】PIT 公告日提取 → backward asof 对齐交易日 → 比率构造
    import dai
    import numpy as np
    import pandas as pd

    LOOKBACK_DAYS = 800

    def _prepare_events(df, value_col):
        if df.empty:
            return df
        out = df.copy()
        out["date"] = pd.to_datetime(out["date"]).dt.normalize()
        out["instrument"] = out["instrument"].astype(str)
        return (
            out.drop_duplicates(subset=["instrument", "date"], keep="last")
            .sort_values(["instrument", "date"])
            .reset_index(drop=True)
        )

    def _asof_align(cal_grp, event_df, value_col):
        cal_grp = cal_grp.copy()
        cal_grp["date"] = pd.to_datetime(cal_grp["date"]).dt.normalize()
        cal_grp["instrument"] = cal_grp["instrument"].astype(str)

        if event_df is None or event_df.empty:
            cal_grp[value_col] = np.nan
            return cal_grp[["date", "instrument", value_col]]

        aligned = pd.merge_asof(
            cal_grp.sort_values("date"),
            event_df[["date", value_col]].sort_values("date"),
            on="date",
            direction="backward",
        )
        return aligned[["date", "instrument", value_col]]

    def _finalize_factor_df(df):
        out = df.copy()
        out["date"] = pd.to_datetime(out["date"]).dt.normalize()
        out["instrument"] = out["instrument"].astype(str)
        out["factor"] = pd.to_numeric(out["factor"], errors="coerce")
        out["factor"] = out["factor"].replace([np.inf, -np.inf], np.nan)
        return out[["date", "instrument", "factor"]]

    def _empty_factor_frame():
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    lookback_start = (start_ts - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")
    fin_end = end_ts.strftime("%Y-%m-%d %H:%M:%S")

    profit_sql = """
    SELECT
        date,
        instrument,
        MAX(
            CASE
                WHEN category = 'ttm'
                THEN net_profit_to_parent_shareholders
            END
        ) AS net_profit_ttm
    FROM bigalpha_2026_financial
    GROUP BY date, instrument
    HAVING net_profit_ttm IS NOT NULL
    """

    equity_sql = """
    SELECT
        date,
        instrument,
        MAX(
            CASE
                WHEN category = 'lf'
                THEN total_equity_to_parent_shareholders
            END
        ) AS equity_lf
    FROM bigalpha_2026_financial
    GROUP BY date, instrument
    HAVING equity_lf IS NOT NULL
      AND ABS(equity_lf) > 1e-8
    """

    profit_df = dai.query(
        profit_sql,
        filters={"date": [lookback_start, fin_end]},
        compression=True,
    ).df()

    equity_df = dai.query(
        equity_sql,
        filters={"date": [lookback_start, fin_end]},
        compression=True,
    ).df()

    cal_df = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    if cal_df.empty:
        return _empty_factor_frame()

    profit_df = _prepare_events(profit_df, "net_profit_ttm")
    equity_df = _prepare_events(equity_df, "equity_lf")

    parts = []
    profit_groups = {k: g for k, g in profit_df.groupby("instrument")} if not profit_df.empty else {}
    equity_groups = {k: g for k, g in equity_df.groupby("instrument")} if not equity_df.empty else {}

    for instrument, cal_grp in cal_df.groupby("instrument", sort=False):
        cal_grp = cal_grp.sort_values("date")
        prof = _asof_align(cal_grp, profit_groups.get(instrument), "net_profit_ttm")
        eq = _asof_align(cal_grp, equity_groups.get(instrument), "equity_lf")
        merged = prof.merge(eq, on=["date", "instrument"], how="left")
        merged["factor"] = merged["net_profit_ttm"] / merged["equity_lf"]
        parts.append(merged[["date", "instrument", "factor"]])

    factor_df = pd.concat(parts, ignore_index=True)
    return _finalize_factor_df(factor_df)
